In [1]:
import os
import numpy as np
import scipy.io
import faiss
import plotly.graph_objects as go


In [7]:
import json

# Open the JSON file
with open('iitd_features_1_with seprate_left_abd_wrute.json', 'r') as file:
    data = json.load(file)  # Load JSON content into a Python dictionary or list

# print(len(data["X"][0]))
print(data["y"])



[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 13, 10013, 10013, 13, 10013, 13, 10013, 13, 13, 10013, 14, 10014, 14, 10014, 10014, 10014, 14, 14, 14, 10014, 15, 15, 10015, 10015, 15, 15, 10015, 10015, 10015, 15, 10016, 16, 16, 16, 10016, 10016, 16, 16, 10016, 10016, 10017, 10017, 10017, 10017, 17, 17, 17, 17, 10017, 17, 18, 18, 10018, 18, 10018, 10018, 18, 10018, 18, 10018, 19, 10019, 10019, 10019, 19, 19, 19, 10019, 10019, 19, 10020, 20, 10020, 20, 10020, 20, 20, 10020, 10020, 20, 21, 21, 10021, 21, 10021, 21, 10021, 21, 10021, 10021, 10022, 22, 22, 10022, 22, 10022, 22, 22, 10022, 10022, 10023, 10023, 23, 23, 10023, 

In [4]:
import json
import numpy as np
import pandas as pd
import faiss
import plotly.graph_objects as go

# ---------------------
# Load JSON data
# ---------------------
def load_data(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    x = np.array(data["X"], dtype='float32')
    y = np.array(data["y"])
    return x, y

# ---------------------
# Split representatives & queries
# ---------------------
def split_representatives(x, y):
    unique_labels = np.unique(y)
    index_vectors = []
    index_labels = []
    query_vectors = []
    query_labels = []

    for label in unique_labels:
        indices = np.where(y == label)[0]
        if len(indices) == 0:
            continue
        # First sample as representative
        index_vectors.append(x[indices[0]])
        index_labels.append(label)
        # Remaining for queries
        if len(indices) > 1:
            remaining = indices[1:]
            query_vectors.extend(x[remaining])
            query_labels.extend(y[remaining])

    return (np.array(index_vectors, dtype='float32'),
            np.array(index_labels),
            np.array(query_vectors, dtype='float32'),
            np.array(query_labels))

# ---------------------
# Build HNSW Index
# ---------------------
def build_hnsw_index(vectors, dim, M=16, efConstruction=50):
    index = faiss.IndexHNSWFlat(dim, M)
    index.hnsw.efConstruction = efConstruction
    index.add(vectors)
    return index

# ---------------------
# Evaluate Hit Rate for varying efSearch and save CSV
# ---------------------
def evaluate_hit_rate(index, index_labels, query_vectors, query_labels, ef_max=100, csv_path=None):
    hit_rates = []
    csv_data = []
    total_indexed = len(index_labels)
    ef_values = range(1, ef_max + 1)

    for ef in ef_values:
        index.hnsw.efSearch = ef
        distances, indices = index.search(query_vectors, k=1)

        # Get predicted labels
        predicted_labels = index_labels[indices.flatten()]

        # Calculate hits
        hits = np.sum(predicted_labels == query_labels)
        total = len(query_labels)
        hit_rate = hits / total if total > 0 else 0

        hit_rates.append(hit_rate)
        penetration = ef / total_indexed
        csv_data.append({"efSearch": ef, "penetration": penetration, "hit_rate": hit_rate})

        print(f"efSearch={ef}, Hit rate={hit_rate:.4f}")

    # Save CSV if path provided
    if csv_path:
        df = pd.DataFrame(csv_data)
        df.to_csv(csv_path, index=False)
        print(f"CSV saved to: {csv_path}")

    return ef_values, hit_rates

# ---------------------
# Plot Hit Rate vs efSearch (using Plotly)
# ---------------------
def plot_hit_rate_plotly(ef_values, hit_rates):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=list(ef_values),
        y=hit_rates,
        mode='lines+markers',
        name="Hit Rate",
        line=dict(width=2),
        marker=dict(size=6)
    ))
    fig.update_layout(
        title="IITDV1_HNSW",
        xaxis_title="efSearch (with Penetration in CSV)",
        yaxis_title="Hit Rate",
        template="plotly_white",
        hovermode="x unified"
    )
    fig.show()

# ---------------------
# Main Workflow
# ---------------------
def main(json_path, ef_max=50, csv_output="hnsw_hit_rate.csv"):
    # Step 1: Load data
    x, y = load_data(json_path)

    # Step 2: Split representatives & queries
    index_x, index_y, query_x, query_y = split_representatives(x, y)

    # Step 3: Build HNSW index
    dim = index_x.shape[1]
    index = build_hnsw_index(index_x, dim)

    # Step 4: Evaluate hit rate and save CSV
    ef_values, hit_rates = evaluate_hit_rate(
        index, index_y, query_x, query_y, ef_max=ef_max, csv_path=csv_output
    )

    # Step 5: Plot graph with Plotly
    plot_hit_rate_plotly(ef_values, hit_rates)

# Example usage
if __name__ == "__main__":
    main("iitd_features_1_with seprate_left_abd_wrute.json", ef_max=100, csv_output="results/IITDV1_HNSW.csv")


efSearch=1, Hit rate=0.6964
efSearch=2, Hit rate=0.8078
efSearch=3, Hit rate=0.8609
efSearch=4, Hit rate=0.8925
efSearch=5, Hit rate=0.9030
efSearch=6, Hit rate=0.9158
efSearch=7, Hit rate=0.9224
efSearch=8, Hit rate=0.9263
efSearch=9, Hit rate=0.9285
efSearch=10, Hit rate=0.9307
efSearch=11, Hit rate=0.9313
efSearch=12, Hit rate=0.9335
efSearch=13, Hit rate=0.9352
efSearch=14, Hit rate=0.9357
efSearch=15, Hit rate=0.9374
efSearch=16, Hit rate=0.9380
efSearch=17, Hit rate=0.9396
efSearch=18, Hit rate=0.9396
efSearch=19, Hit rate=0.9396
efSearch=20, Hit rate=0.9402
efSearch=21, Hit rate=0.9402
efSearch=22, Hit rate=0.9402
efSearch=23, Hit rate=0.9407
efSearch=24, Hit rate=0.9407
efSearch=25, Hit rate=0.9418
efSearch=26, Hit rate=0.9424
efSearch=27, Hit rate=0.9424
efSearch=28, Hit rate=0.9424
efSearch=29, Hit rate=0.9424
efSearch=30, Hit rate=0.9429
efSearch=31, Hit rate=0.9429
efSearch=32, Hit rate=0.9429
efSearch=33, Hit rate=0.9429
efSearch=34, Hit rate=0.9429
efSearch=35, Hit rate=0

In [1]:
import json
import numpy as np
import pandas as pd
import faiss
import plotly.graph_objects as go

# ---------------------
# Load JSON data
# ---------------------
def load_data(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    x = np.array(data["X"], dtype='float32')
    y = np.array(data["y"])
    return x, y

# ---------------------
# Split representatives & queries
# ---------------------
def split_representatives(x, y):
    unique_labels = np.unique(y)
    index_vectors, index_labels = [], []
    query_vectors, query_labels = [], []

    for label in unique_labels:
        indices = np.where(y == label)[0]
        if len(indices) == 0:
            continue
        # First one → representative
        index_vectors.append(x[indices[0]])
        index_labels.append(label)
        # Remaining → queries
        if len(indices) > 1:
            remaining = indices[1:]
            query_vectors.extend(x[remaining])
            query_labels.extend(y[remaining])

    return (np.array(index_vectors, dtype='float32'),
            np.array(index_labels),
            np.array(query_vectors, dtype='float32'),
            np.array(query_labels))

# ---------------------
# Build HNSW Index
# ---------------------
def build_hnsw_index(vectors, dim, M=16, efConstruction=50):
    index = faiss.IndexHNSWFlat(dim, M)
    index.hnsw.efConstruction = efConstruction
    index.add(vectors)
    return index

# ---------------------
# Evaluate Hit Rate for varying top-k (ef fixed)
# ---------------------
def evaluate_hit_rate_topk(index, index_labels, query_vectors, query_labels, ef=20, k_max=50, csv_path=None):
    hit_rates = []
    csv_data = []
    total_indexed = len(index_labels)
    k_values = range(1, k_max + 1)

    index.hnsw.efSearch = ef

    # Search with max-k once (to reuse results efficiently)
    distances, indices = index.search(query_vectors, k=k_max)

    for k in k_values:
        # Take top-k predictions
        topk_indices = indices[:, :k]
        topk_labels = index_labels[topk_indices]

        # A query is a hit if its true label appears in top-k retrieved labels
        hits = np.sum([
            query_labels[i] in topk_labels[i]
            for i in range(len(query_labels))
        ])

        total = len(query_labels)
        hit_rate = hits / total if total > 0 else 0

        penetration = k / total_indexed
        csv_data.append({"k": k, "penetration": penetration, "hit_rate": hit_rate})
        hit_rates.append(hit_rate)

        print(f"k={k}, Hit rate={hit_rate:.4f}, Penetration={penetration:.4f}")

    # Save CSV if requested
    if csv_path:
        df = pd.DataFrame(csv_data)
        df.to_csv(csv_path, index=False)
        print(f"CSV saved to: {csv_path}")

    return k_values, hit_rates

# ---------------------
# Plot Hit Rate vs Top-k (with Penetration)
# ---------------------
def plot_hit_rate_plotly_topk(k_values, hit_rates, total_indexed):
    penetration_rates = [k / total_indexed for k in k_values]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=penetration_rates,
        y=hit_rates,
        mode='lines+markers',
        name="Hit Rate",
        line=dict(width=2),
        marker=dict(size=6)
    ))
    fig.update_layout(
        title="IITDV1_HNSW (ef=20, varying top-k)",
        xaxis_title="Penetration Rate (k / total indexed)",
        yaxis_title="Hit Rate",
        template="plotly_white",
        hovermode="x unified"
    )
    fig.show()

# ---------------------
# Main Workflow
# ---------------------
def main(json_path, ef=20, k_max=50, csv_output="hnsw_hit_rate_topk.csv"):
    # Step 1: Load data
    x, y = load_data(json_path)

    # Step 2: Split representatives & queries
    index_x, index_y, query_x, query_y = split_representatives(x, y)

    # Step 3: Build HNSW index
    dim = index_x.shape[1]
    index = build_hnsw_index(index_x, dim)

    # Step 4: Evaluate hit rate varying top-k
    k_values, hit_rates = evaluate_hit_rate_topk(
        index, index_y, query_x, query_y, ef=ef, k_max=k_max, csv_path=csv_output
    )

    # Step 5: Plot graph
    plot_hit_rate_plotly_topk(k_values, hit_rates, len(index_y))

# Example usage
if __name__ == "__main__":
    main("iitd_features_1_with seprate_left_abd_wrute.json",
         ef=20,
         k_max=70,
         csv_output="results/IITDV1_HNSW_TopK.csv")


k=1, Hit rate=0.9402, Penetration=0.0023
k=2, Hit rate=0.9501, Penetration=0.0046
k=3, Hit rate=0.9551, Penetration=0.0069
k=4, Hit rate=0.9584, Penetration=0.0092
k=5, Hit rate=0.9607, Penetration=0.0115
k=6, Hit rate=0.9618, Penetration=0.0138
k=7, Hit rate=0.9618, Penetration=0.0161
k=8, Hit rate=0.9634, Penetration=0.0184
k=9, Hit rate=0.9634, Penetration=0.0207
k=10, Hit rate=0.9634, Penetration=0.0230
k=11, Hit rate=0.9645, Penetration=0.0253
k=12, Hit rate=0.9645, Penetration=0.0276
k=13, Hit rate=0.9645, Penetration=0.0299
k=14, Hit rate=0.9645, Penetration=0.0322
k=15, Hit rate=0.9651, Penetration=0.0345
k=16, Hit rate=0.9651, Penetration=0.0368
k=17, Hit rate=0.9651, Penetration=0.0391
k=18, Hit rate=0.9651, Penetration=0.0414
k=19, Hit rate=0.9662, Penetration=0.0437
k=20, Hit rate=0.9668, Penetration=0.0460
k=21, Hit rate=0.9684, Penetration=0.0483
k=22, Hit rate=0.9684, Penetration=0.0506
k=23, Hit rate=0.9684, Penetration=0.0529
k=24, Hit rate=0.9684, Penetration=0.0552
k

In [ ]:
import json
import numpy as np
import faiss
import plotly.graph_objects as go

# ---------------------
# Load JSON data
# ---------------------
def load_data(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    x = np.array(data["X"], dtype='float32')
    y = np.array(data["y"])
    return x, y

# ---------------------
# Split representatives & queries
# ---------------------
def split_representatives(x, y):
    unique_labels = np.unique(y)
    index_vectors = []
    index_labels = []
    query_vectors = []
    query_labels = []

    for label in unique_labels:
        indices = np.where(y == label)[0]
        if len(indices) == 0:
            continue
        index_vectors.append(x[indices[0]])
        index_labels.append(label)
        if len(indices) > 1:
            remaining = indices[1:]
            query_vectors.extend(x[remaining])
            query_labels.extend(y[remaining])

    return (np.array(index_vectors, dtype='float32'),
            np.array(index_labels),
            np.array(query_vectors, dtype='float32'),
            np.array(query_labels))

# ---------------------
# Build HNSW Index
# ---------------------
def build_hnsw_index(vectors, dim, M=16, efConstruction=200):
    index = faiss.IndexHNSWFlat(dim, M)
    index.hnsw.efConstruction = efConstruction
    index.add(vectors)
    return index

# ---------------------
# Evaluate Hit Rate and Label X-Axis with Penetration
# ---------------------
def evaluate_hit_rate(index, index_labels, query_vectors, query_labels, ef_max=100):
    hit_rates = []
    x_labels = []
    total_indexed = len(index_labels)
    highlight_point = None

    for ef in range(1, ef_max + 1):
        index.hnsw.efSearch = ef
        distances, indices = index.search(query_vectors, k=1)

        predicted_labels = index_labels[indices.flatten()]
        hits = np.sum(predicted_labels == query_labels)
        total = len(query_labels)
        hit_rate = hits / total if total > 0 else 0
        hit_rates.append(hit_rate)

        penetration = ef / total_indexed
        x_labels.append(f"{ef} ({penetration:.2f})")

        if highlight_point is None and hit_rate >= 0.98:
            highlight_point = (ef, hit_rate, penetration)

        print(f"efSearch={ef}, Hit rate={hit_rate:.4f}")

    return x_labels, hit_rates, highlight_point

# ---------------------
# Plot Hit Rate with Penetration Labels
# ---------------------
def plot_hit_rate_plotly(x_labels, hit_rates, highlight_point):
    fig = go.Figure()

    # Main line
    fig.add_trace(go.Scatter(
        x=x_labels,
        y=hit_rates,
        mode='lines+markers',
        name="Hit Rate",
        line=dict(width=2),
        marker=dict(size=6)
    ))

    # Highlight first ≥99% hit rate
    if highlight_point:
        ef_highlight, hr_highlight, penetration = highlight_point
        highlight_label = f"{ef_highlight} ({penetration:.2f})"
        fig.add_trace(go.Scatter(
            x=[highlight_label],
            y=[hr_highlight],
            mode='markers+text',
            marker=dict(color='red', size=12, symbol='star'),
            text=[f"Penetration={penetration:.2f}"],
            textposition="top center",
            name="First ≥99% Hit"
        ))

    fig.update_layout(
        title="IITDV1_HNSW",
        xaxis_title="efSearch (Penetration)",
        yaxis_title="Hit Rate",
        template="plotly_white",
        hovermode="x unified"
    )
    fig.show()

# ---------------------
# Main Workflow
# ---------------------
def main(json_path, ef_max=50):
    x, y = load_data(json_path)
    index_x, index_y, query_x, query_y = split_representatives(x, y)
    dim = index_x.shape[1]
    index = build_hnsw_index(index_x, dim)

    x_labels, hit_rates, highlight_point = evaluate_hit_rate(
        index, index_y, query_x, query_y, ef_max=ef_max
    )

    plot_hit_rate_plotly(x_labels, hit_rates, highlight_point)

# Example usage
if __name__ == "__main__":
    main("iitd_features_1_with seprate_left_abd_wrute.json", ef_max=50)


efSearch=1, Hit rate=0.7302
efSearch=2, Hit rate=0.8360
efSearch=3, Hit rate=0.8787
efSearch=4, Hit rate=0.9047
efSearch=5, Hit rate=0.9136
efSearch=6, Hit rate=0.9224
efSearch=7, Hit rate=0.9269
efSearch=8, Hit rate=0.9291
efSearch=9, Hit rate=0.9319
efSearch=10, Hit rate=0.9341
efSearch=11, Hit rate=0.9363
efSearch=12, Hit rate=0.9380
efSearch=13, Hit rate=0.9396
efSearch=14, Hit rate=0.9396
efSearch=15, Hit rate=0.9396
efSearch=16, Hit rate=0.9396
efSearch=17, Hit rate=0.9396
efSearch=18, Hit rate=0.9407
efSearch=19, Hit rate=0.9413
efSearch=20, Hit rate=0.9413
efSearch=21, Hit rate=0.9413
efSearch=22, Hit rate=0.9413
efSearch=23, Hit rate=0.9413
efSearch=24, Hit rate=0.9413
efSearch=25, Hit rate=0.9424
efSearch=26, Hit rate=0.9429
efSearch=27, Hit rate=0.9429
efSearch=28, Hit rate=0.9429
efSearch=29, Hit rate=0.9429
efSearch=30, Hit rate=0.9429
efSearch=31, Hit rate=0.9429
efSearch=32, Hit rate=0.9429
efSearch=33, Hit rate=0.9429
efSearch=34, Hit rate=0.9429
efSearch=35, Hit rate=0

In [16]:
def get_incorrect_indices(index, index_labels, query_vectors, query_labels, ef_search=50):
    """
    Returns the indices of queries where predicted label != actual label
    for a given ef_search value.
    """
    # Set efSearch
    index.hnsw.efSearch = ef_search
    
    # Perform top-1 search
    distances, indices = index.search(query_vectors, k=1)
    
    predicted_labels = index_labels[indices.flatten()]
    incorrect = np.where(predicted_labels != query_labels)[0]
    
    return incorrect, predicted_labels[incorrect], query_labels[incorrect]


In [19]:
def main_check_incorrect(json_path, ef_search=50):
    # Load and prepare data
    x, y = load_data(json_path)
    index_x, index_y, query_x, query_y = split_representatives(x, y)
    dim = index_x.shape[1]
    index = build_hnsw_index(index_x, dim)

    # Get incorrect matches
    incorrect_indices, predicted, actual = get_incorrect_indices(
        index, index_y, query_x, query_y, ef_search=ef_search
    )

    print(len(index_x))
    print(f"Total queries: {len(query_y)}")
    print(f"Incorrect matches: {len(incorrect_indices)}")
    print("Indices of incorrect queries:", incorrect_indices)
    print("Predicted labels:", predicted)
    print("Actual labels:", actual)

# Example usage
if __name__ == "__main__":
    main_check_incorrect("iitd_features_1_with seprate_left_abd_wrute.json", ef_search=50)


435
Total queries: 1805
Incorrect matches: 103
Indices of incorrect queries: [   2   13   16   27   33  101  103  105  111  137  144  161  165  166
  167  168  169  170  173  233  290  340  341  342  343  392  394  412
  413  414  415  428  429  430  431  444  493  497  544  556  614  836
  837  838  839  869  870  871  967 1004 1005 1007 1014 1035 1036 1056
 1059 1060 1061 1082 1113 1116 1156 1179 1189 1190 1194 1282 1318 1319
 1321 1322 1323 1326 1365 1371 1372 1373 1374 1375 1406 1409 1431 1449
 1451 1452 1485 1486 1487 1560 1590 1592 1613 1614 1615 1616 1630 1632
 1690 1693 1695 1696 1745]
Predicted labels: [  198   189 10054    69   197 10192 10192 10194    77 10018 10019 10081
   105   105   170   105 10054 10211   223 10189 10211    36    36    36
    36    70 10081    77    98   120   220   159 10090 10046   212    86
   187 10072 10064   157 10019    39   190   153   153   205   205   205
 10100    23 10091 10032   159   120 10098 10083    37    48 10221    70
    50 10217    

In [ ]:
### FLAT

In [2]:
import json
import numpy as np
import pandas as pd
import faiss
import plotly.graph_objects as go

# ---------------------
# Load JSON data
# ---------------------
def load_data(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    x = np.array(data["X"], dtype='float32')
    y = np.array(data["y"])
    return x, y

# ---------------------
# Split representatives & queries
# ---------------------
def split_representatives(x, y):
    unique_labels = np.unique(y)
    index_vectors = []
    index_labels = []
    query_vectors = []
    query_labels = []

    for label in unique_labels:
        indices = np.where(y == label)[0]
        if len(indices) == 0:
            continue
        index_vectors.append(x[indices[0]])
        index_labels.append(label)
        if len(indices) > 1:
            remaining = indices[1:]
            query_vectors.extend(x[remaining])
            query_labels.extend(y[remaining])

    return (np.array(index_vectors, dtype='float32'),
            np.array(index_labels),
            np.array(query_vectors, dtype='float32'),
            np.array(query_labels))

# ---------------------
# Build FLAT Index (Euclidean)
# ---------------------
def build_flat_index(vectors, dim):
    index = faiss.IndexFlatL2(dim)  # Euclidean distance
    index.add(vectors)
    return index

# ---------------------
# Evaluate Hit Rate vs top-k and Save to CSV
# ---------------------
def evaluate_hit_rate_topk(index, index_labels, query_vectors, query_labels, k_max=50, csv_path=None):
    hit_rates = []
    x_labels = []
    csv_data = []
    total_indexed = len(index_labels)
    highlight_point = None

    for k in range(1, k_max + 1):
        distances, indices = index.search(query_vectors, k=k)

        predicted_labels_k = index_labels[indices]
        hits = sum(query_labels[i] in predicted_labels_k[i] for i in range(len(query_labels)))
        total = len(query_labels)
        hit_rate = hits / total if total > 0 else 0
        hit_rates.append(hit_rate)

        penetration = k / total_indexed
        x_labels.append(f"{k} ({penetration:.2f})")
        csv_data.append({"k": k, "penetration": penetration, "hit_rate": hit_rate})

        if highlight_point is None and hit_rate >= 0.99:
            highlight_point = (k, hit_rate, penetration)

        print(f"top-k={k}, Hit rate={hit_rate:.4f}")

    # Save CSV if path provided
    if csv_path:
        df = pd.DataFrame(csv_data)
        df.to_csv(csv_path, index=False)
        print(f"CSV saved to: {csv_path}")

    return x_labels, hit_rates, highlight_point

# ---------------------
# Plot Hit Rate vs top-k
# ---------------------
def plot_hit_rate_plotly(x_labels, hit_rates, highlight_point):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=x_labels,
        y=hit_rates,
        mode='lines+markers',
        name="Hit Rate",
        line=dict(width=2),
        marker=dict(size=6)
    ))

    if highlight_point:
        k_highlight, hr_highlight, penetration = highlight_point
        highlight_label = f"{k_highlight} ({penetration:.2f})"
        fig.add_trace(go.Scatter(
            x=[highlight_label],
            y=[hr_highlight],
            mode='markers+text',
            marker=dict(color='red', size=12, symbol='star'),
            text=[f"Penetration={penetration:.2f}"],
            textposition="top center",
            name="First ≥99% Hit"
        ))

    fig.update_layout(
        title="IITDV1_FLAT (Top-k Hit Rate)",
        xaxis_title="Top-k (Penetration)",
        yaxis_title="Hit Rate",
        template="plotly_white",
        hovermode="x unified"
    )
    fig.show()

# ---------------------
# Main Workflow
# ---------------------
def main(json_path, k_max=50, csv_output="hit_rate_topk.csv"):
    x, y = load_data(json_path)
    index_x, index_y, query_x, query_y = split_representatives(x, y)
    dim = index_x.shape[1]
    index = build_flat_index(index_x, dim)

    x_labels, hit_rates, highlight_point = evaluate_hit_rate_topk(
        index, index_y, query_x, query_y, k_max=k_max, csv_path=csv_output
    )

    plot_hit_rate_plotly(x_labels, hit_rates, highlight_point)

# Example usage
if __name__ == "__main__":
    main("iitd_features_1_with seprate_left_abd_wrute.json", k_max=100, csv_output="results/IITDV1_FLAT.csv")



top-k=1, Hit rate=0.9429
top-k=2, Hit rate=0.9529
top-k=3, Hit rate=0.9584
top-k=4, Hit rate=0.9629
top-k=5, Hit rate=0.9645
top-k=6, Hit rate=0.9662
top-k=7, Hit rate=0.9668
top-k=8, Hit rate=0.9684
top-k=9, Hit rate=0.9684
top-k=10, Hit rate=0.9684
top-k=11, Hit rate=0.9695
top-k=12, Hit rate=0.9701
top-k=13, Hit rate=0.9706
top-k=14, Hit rate=0.9706
top-k=15, Hit rate=0.9712
top-k=16, Hit rate=0.9712
top-k=17, Hit rate=0.9712
top-k=18, Hit rate=0.9717
top-k=19, Hit rate=0.9723
top-k=20, Hit rate=0.9734
top-k=21, Hit rate=0.9740
top-k=22, Hit rate=0.9751
top-k=23, Hit rate=0.9751
top-k=24, Hit rate=0.9751
top-k=25, Hit rate=0.9751
top-k=26, Hit rate=0.9756
top-k=27, Hit rate=0.9756
top-k=28, Hit rate=0.9756
top-k=29, Hit rate=0.9767
top-k=30, Hit rate=0.9767
top-k=31, Hit rate=0.9767
top-k=32, Hit rate=0.9767
top-k=33, Hit rate=0.9778
top-k=34, Hit rate=0.9784
top-k=35, Hit rate=0.9789
top-k=36, Hit rate=0.9789
top-k=37, Hit rate=0.9789
top-k=38, Hit rate=0.9801
top-k=39, Hit rate=0.

In [ ]:
### IVF

In [14]:
import json
import numpy as np
import pandas as pd
import faiss
import plotly.graph_objects as go

# ---------------------
# Load JSON data
# ---------------------
def load_data(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    x = np.array(data["X"], dtype='float32')
    y = np.array(data["y"])
    return x, y

# ---------------------
# Split representatives & queries
# ---------------------
def split_representatives(x, y):
    unique_labels = np.unique(y)
    index_vectors = []
    index_labels = []
    query_vectors = []
    query_labels = []

    for label in unique_labels:
        indices = np.where(y == label)[0]
        if len(indices) == 0:
            continue
        index_vectors.append(x[indices[0]])
        index_labels.append(label)
        if len(indices) > 1:
            remaining = indices[1:]
            query_vectors.extend(x[remaining])
            query_labels.extend(y[remaining])

    return (np.array(index_vectors, dtype='float32'),
            np.array(index_labels),
            np.array(query_vectors, dtype='float32'),
            np.array(query_labels))

# ---------------------
# Build IVF Index (Euclidean)
# ---------------------
def build_ivf_index(vectors, dim, nlist):
    quantizer = faiss.IndexFlatL2(dim)
    index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_L2)
    index.train(vectors)
    index.add(vectors)
    return index

# ---------------------
# Evaluate Hit Rate (k=1) vs (nprobe, nlist)
# ---------------------
def evaluate_hit_rate_clusters(index_vectors, index_labels, query_vectors, query_labels,
                               nlist_values, csv_path=None):
    results = []
    k = 1  # Fixed top-1

    for nlist in nlist_values:
        index = build_ivf_index(index_vectors, index_vectors.shape[1], nlist=nlist)

        for nprobe in range(1, nlist + 1):
            index.nprobe = nprobe
            distances, indices = index.search(query_vectors, k=k)
            predicted_labels = index_labels[indices]
            hits = sum(query_labels[i] in predicted_labels[i] for i in range(len(query_labels)))
            total = len(query_labels)
            hit_rate = hits / total if total > 0 else 0
            penetration = nprobe / nlist

            results.append({
                "nlist": nlist,
                "nprobe": nprobe,
                "penetration": penetration,
                "hit_rate": hit_rate
            })

            # print(f"nlist={nlist}, nprobe={nprobe}, penetration={penetration:.2f}, hit_rate={hit_rate:.4f}")

    df = pd.DataFrame(results)
    if csv_path:
        df.to_csv(csv_path, index=False)
        print(f"CSV saved to: {csv_path}")
    return df

# ---------------------
# Plot Heatmap (nprobe vs nlist) with Penetration
# ---------------------
def plot_hit_rate_heatmap(df):
    # Create pivot tables
    pivot_hit_rate = df.pivot(index="nlist", columns="nprobe", values="hit_rate")
    pivot_penetration = df.pivot(index="nlist", columns="nprobe", values="penetration")

    # Align both pivots to same shape & sort
    pivot_hit_rate = pivot_hit_rate.sort_index(axis=0).sort_index(axis=1)
    pivot_penetration = pivot_penetration.reindex_like(pivot_hit_rate)

    # Replace NaN with 0 (or any default)
    pivot_hit_rate = pivot_hit_rate.fillna(0)
    pivot_penetration = pivot_penetration.fillna(0)

    fig = go.Figure(data=go.Heatmap(
        z=pivot_hit_rate.values,
        x=pivot_hit_rate.columns,
        y=pivot_hit_rate.index,
        colorscale="Viridis",
        colorbar=dict(title="Hit Rate"),
        customdata=pivot_penetration.values.astype(float),  # Ensure float
        hovertemplate=(
            "Total Clusters (nlist): %{y}<br>"
            "Clusters Searched (nprobe): %{x}<br>"
            "Hit Rate: %{z:.4f}<br>"
            "Penetration (nprobe/nlist): %{customdata}<extra></extra>"
        )
    ))

    fig.update_layout(
        title="Top-1 Hit Rate: Clusters Searched vs Total Clusters",
        xaxis_title="Clusters Searched (nprobe)",
        yaxis_title="Total Clusters (nlist)",
        template="plotly_white"
    )
    fig.show()



# ---------------------
# Main Workflow
# ---------------------
def main(json_path, csv_output="hit_rate_top1_clusters.csv"):
    nlist_values = list(range(1, 51))  # nlist = 1 to 50
    x, y = load_data(json_path)
    index_x, index_y, query_x, query_y = split_representatives(x, y)

    df = evaluate_hit_rate_clusters(
        index_x, index_y, query_x, query_y,
        nlist_values=nlist_values, csv_path=csv_output
    )

    plot_hit_rate_heatmap(df)

# Example usage
if __name__ == "__main__":
    main("iitd_features_1_with seprate_left_abd_wrute.json",
         csv_output="results/IITDV1_IVF_top1_clusters.csv")


WARNING clustering 435 points to 12 centroids: please provide at least 468 training points
WARNING clustering 435 points to 13 centroids: please provide at least 507 training points
WARNING clustering 435 points to 14 centroids: please provide at least 546 training points
WARNING clustering 435 points to 15 centroids: please provide at least 585 training points
WARNING clustering 435 points to 16 centroids: please provide at least 624 training points
WARNING clustering 435 points to 17 centroids: please provide at least 663 training points
WARNING clustering 435 points to 18 centroids: please provide at least 702 training points
WARNING clustering 435 points to 19 centroids: please provide at least 741 training points
WARNING clustering 435 points to 20 centroids: please provide at least 780 training points
WARNING clustering 435 points to 21 centroids: please provide at least 819 training points
WARNING clustering 435 points to 22 centroids: please provide at least 858 training points

CSV saved to: results/IITDV1_IVF_top1_clusters.csv


In [1]:
import json
import numpy as np
import faiss
import plotly.graph_objects as go

# ---------------------
# Load JSON data
# ---------------------
def load_data(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    x = np.array(data["X"], dtype='float32')
    y = np.array(data["y"])
    return x, y

# ---------------------
# Split representatives & queries
# ---------------------
def split_representatives(x, y):
    unique_labels = np.unique(y)
    index_vectors = []
    index_labels = []
    query_vectors = []
    query_labels = []

    for label in unique_labels:
        indices = np.where(y == label)[0]
        if len(indices) == 0:
            continue
        index_vectors.append(x[indices[0]])
        index_labels.append(label)
        if len(indices) > 1:
            remaining = indices[1:]
            query_vectors.extend(x[remaining])
            query_labels.extend(y[remaining])

    return (np.array(index_vectors, dtype='float32'),
            np.array(index_labels),
            np.array(query_vectors, dtype='float32'),
            np.array(query_labels))

# ---------------------
# Build IVF Index
# ---------------------
def build_ivf_index(vectors, dim, nlist=14):
    quantizer = faiss.IndexFlatL2(dim)  # base quantizer
    index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_L2)
    index.train(vectors)   # train on representative vectors
    index.add(vectors)     # add all vectors
    return index

# ---------------------
# Evaluate Hit Rate (varying nprobe)
# ---------------------
def evaluate_hit_rate_ivf(index, index_labels, query_vectors, query_labels, nlist=14):
    hit_rates = []
    x_labels = []
    highlight_point = None
    total_indexed = len(index_labels)

    for nprobe in range(1, nlist + 1):
        index.nprobe = nprobe
        distances, indices = index.search(query_vectors, k=1)
        predicted_labels = index_labels[indices.flatten()]
        hits = np.sum(predicted_labels == query_labels)
        total = len(query_labels)
        hit_rate = hits / total if total > 0 else 0
        hit_rates.append(hit_rate)

        penetration = nprobe / nlist
        x_labels.append(f"{nprobe} ({penetration:.2f})")

        if highlight_point is None and hit_rate >= 0.98:
            highlight_point = (nprobe, hit_rate, penetration)

        print(f"nprobe={nprobe}, Hit rate={hit_rate:.4f}")

    return x_labels, hit_rates, highlight_point

# ---------------------
# Plot Hit Rate vs Penetration
# ---------------------
def plot_hit_rate_plotly(x_labels, hit_rates, highlight_point):
    fig = go.Figure()

    # Main line
    fig.add_trace(go.Scatter(
        x=x_labels,
        y=hit_rates,
        mode='lines+markers',
        name="Hit Rate",
        line=dict(width=2),
        marker=dict(size=6)
    ))

    # Highlight first ≥98% hit rate
    if highlight_point:
        nprobe_highlight, hr_highlight, penetration = highlight_point
        highlight_label = f"{nprobe_highlight} ({penetration:.2f})"
        fig.add_trace(go.Scatter(
            x=[highlight_label],
            y=[hr_highlight],
            mode='markers+text',
            marker=dict(color='red', size=12, symbol='star'),
            text=[f"Penetration={penetration:.2f}"],
            textposition="top center",
            name="First ≥98% Hit"
        ))

    fig.update_layout(
        title="IITDV1_IVF",
        xaxis_title="nprobe (Penetration = nprobe/nlist)",
        yaxis_title="Hit Rate",
        template="plotly_white",
        hovermode="x unified"
    )
    fig.show()

# ---------------------
# Main Workflow
# ---------------------
def main(json_path, nlist=14):
    x, y = load_data(json_path)
    index_x, index_y, query_x, query_y = split_representatives(x, y)
    dim = index_x.shape[1]
    index = build_ivf_index(index_x, dim, nlist=nlist)

    x_labels, hit_rates, highlight_point = evaluate_hit_rate_ivf(
        index, index_y, query_x, query_y, nlist=nlist
    )

    plot_hit_rate_plotly(x_labels, hit_rates, highlight_point)

# Example usage
if __name__ == "__main__":
    main("iitd_features_1_with seprate_left_abd_wrute.json", nlist=14)


WARNING clustering 435 points to 14 centroids: please provide at least 546 training points


nprobe=1, Hit rate=0.7180
nprobe=2, Hit rate=0.8781
nprobe=3, Hit rate=0.9102
nprobe=4, Hit rate=0.9324
nprobe=5, Hit rate=0.9396
nprobe=6, Hit rate=0.9418
nprobe=7, Hit rate=0.9418
nprobe=8, Hit rate=0.9418
nprobe=9, Hit rate=0.9424
nprobe=10, Hit rate=0.9429
nprobe=11, Hit rate=0.9429
nprobe=12, Hit rate=0.9429
nprobe=13, Hit rate=0.9429
nprobe=14, Hit rate=0.9429


In [4]:
import json
import numpy as np
import faiss
import plotly.graph_objects as go

# ---------------------
# Load JSON data
# ---------------------
def load_data(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    x = np.array(data["X"], dtype='float32')
    y = np.array(data["y"])
    return x, y

# ---------------------
# Split representatives & queries
# ---------------------
def split_representatives(x, y):
    unique_labels = np.unique(y)
    index_vectors = []
    index_labels = []
    query_vectors = []
    query_labels = []

    for label in unique_labels:
        indices = np.where(y == label)[0]
        if len(indices) == 0:
            continue
        index_vectors.append(x[indices[0]])
        index_labels.append(label)
        if len(indices) > 1:
            remaining = indices[1:]
            query_vectors.extend(x[remaining])
            query_labels.extend(y[remaining])

    return (np.array(index_vectors, dtype='float32'),
            np.array(index_labels),
            np.array(query_vectors, dtype='float32'),
            np.array(query_labels))

# ---------------------
# Build LSH Index
# ---------------------
def build_lsh_index(vectors, nbits=16):
    dim = vectors.shape[1]
    index = faiss.IndexLSH(dim, nbits)
    index.add(vectors)
    return index

# ---------------------
# Evaluate Hit Rate (varying top_k)
# ---------------------
def evaluate_hit_rate_lsh(index, index_labels, query_vectors, query_labels, top_k_max=50):
    hit_rates = []
    x_labels = []
    highlight_point = None
    total_indexed = len(index_labels)

    for k in range(1, top_k_max + 1):
        distances, indices = index.search(query_vectors, k=k)
        
        # Check if top_k contains the correct label
        hits = 0
        for q_idx, q_label in enumerate(query_labels):
            retrieved_labels = index_labels[indices[q_idx]]
            if q_label in retrieved_labels:
                hits += 1

        total = len(query_labels)
        hit_rate = hits / total if total > 0 else 0
        hit_rates.append(hit_rate)

        penetration = k / total_indexed
        x_labels.append(f"{k} ({penetration:.2f})")

        if highlight_point is None and hit_rate >= 0.98:
            highlight_point = (k, hit_rate, penetration)

        print(f"top_k={k}, Hit rate={hit_rate:.4f}")

    return x_labels, hit_rates, highlight_point

# ---------------------
# Plot Hit Rate vs Penetration
# ---------------------
def plot_hit_rate_plotly(x_labels, hit_rates, highlight_point):
    fig = go.Figure()

    # Main line
    fig.add_trace(go.Scatter(
        x=x_labels,
        y=hit_rates,
        mode='lines+markers',
        name="Hit Rate",
        line=dict(width=2),
        marker=dict(size=6)
    ))

    # Highlight first ≥98% hit rate
    if highlight_point:
        k_highlight, hr_highlight, penetration = highlight_point
        highlight_label = f"{k_highlight} ({penetration:.2f})"
        fig.add_trace(go.Scatter(
            x=[highlight_label],
            y=[hr_highlight],
            mode='markers+text',
            marker=dict(color='red', size=12, symbol='star'),
            text=[f"Penetration={penetration:.2f}"],
            textposition="top center",
            name="First ≥98% Hit"
        ))

    fig.update_layout(
        title="IITDV1_LSH",
        xaxis_title="top_k (Penetration = k / total indexed points)",
        yaxis_title="Hit Rate",
        template="plotly_white",
        hovermode="x unified"
    )
    fig.show()

# ---------------------
# Main Workflow
# ---------------------
def main(json_path, nbits=16, top_k_max=50):
    x, y = load_data(json_path)
    index_x, index_y, query_x, query_y = split_representatives(x, y)
    
    index = build_lsh_index(index_x, nbits=nbits)

    x_labels, hit_rates, highlight_point = evaluate_hit_rate_lsh(
        index, index_y, query_x, query_y, top_k_max=top_k_max
    )

    plot_hit_rate_plotly(x_labels, hit_rates, highlight_point)

# Example usage
if __name__ == "__main__":
    main("iitd_features_1_with seprate_left_abd_wrute.json", nbits=16, top_k_max=200)


top_k=1, Hit rate=0.1501
top_k=2, Hit rate=0.2066
top_k=3, Hit rate=0.2626
top_k=4, Hit rate=0.2997
top_k=5, Hit rate=0.3302
top_k=6, Hit rate=0.3518
top_k=7, Hit rate=0.3734
top_k=8, Hit rate=0.3950
top_k=9, Hit rate=0.4122
top_k=10, Hit rate=0.4332
top_k=11, Hit rate=0.4532
top_k=12, Hit rate=0.4681
top_k=13, Hit rate=0.4825
top_k=14, Hit rate=0.4958
top_k=15, Hit rate=0.5069
top_k=16, Hit rate=0.5163
top_k=17, Hit rate=0.5269
top_k=18, Hit rate=0.5374
top_k=19, Hit rate=0.5452
top_k=20, Hit rate=0.5518
top_k=21, Hit rate=0.5612
top_k=22, Hit rate=0.5684
top_k=23, Hit rate=0.5801
top_k=24, Hit rate=0.5834
top_k=25, Hit rate=0.5900
top_k=26, Hit rate=0.6017
top_k=27, Hit rate=0.6072
top_k=28, Hit rate=0.6116
top_k=29, Hit rate=0.6166
top_k=30, Hit rate=0.6244
top_k=31, Hit rate=0.6321
top_k=32, Hit rate=0.6366
top_k=33, Hit rate=0.6393
top_k=34, Hit rate=0.6460
top_k=35, Hit rate=0.6515
top_k=36, Hit rate=0.6548
top_k=37, Hit rate=0.6593
top_k=38, Hit rate=0.6609
top_k=39, Hit rate=0.

In [7]:
import json
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min
import plotly.graph_objects as go

# ---------------------
# Load JSON data
# ---------------------
def load_data(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    x = np.array(data["X"], dtype='float32')
    y = np.array(data["y"])
    return x, y

# ---------------------
# Split representatives & queries
# ---------------------
def split_representatives(x, y):
    unique_labels = np.unique(y)
    index_vectors = []
    index_labels = []
    query_vectors = []
    query_labels = []

    for label in unique_labels:
        indices = np.where(y == label)[0]
        if len(indices) == 0:
            continue
        index_vectors.append(x[indices[0]])
        index_labels.append(label)
        if len(indices) > 1:
            remaining = indices[1:]
            query_vectors.extend(x[remaining])
            query_labels.extend(y[remaining])

    return (np.array(index_vectors, dtype='float32'),
            np.array(index_labels),
            np.array(query_vectors, dtype='float32'),
            np.array(query_labels))

# ---------------------
# Build KMeans clusters
# ---------------------
def build_knn_clusters(vectors, n_clusters=14):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans.fit(vectors)
    cluster_assignments = kmeans.labels_
    clusters = {i: [] for i in range(n_clusters)}
    for idx, c in enumerate(cluster_assignments):
        clusters[c].append(idx)
    return kmeans, clusters

# ---------------------
# Evaluate hit rate (varying clusters searched)
# ---------------------
def evaluate_hit_rate_knn(kmeans, clusters, index_vectors, index_labels, query_vectors, query_labels, max_clusters=None):
    hit_rates = []
    x_labels = []
    highlight_point = None
    total_indexed = len(index_labels)
    
    n_clusters = len(clusters)
    if max_clusters is None:
        max_clusters = n_clusters

    for clusters_to_search in range(1, max_clusters + 1):
        hits = 0
        total_searched_vectors = 0

        for q_idx, q_vec in enumerate(query_vectors):
            # Distance to all centroids
            centroid_dists = np.linalg.norm(kmeans.cluster_centers_ - q_vec, axis=1)
            nearest_clusters = np.argsort(centroid_dists)[:clusters_to_search]

            # Collect candidate indices
            candidate_indices = []
            for c in nearest_clusters:
                candidate_indices.extend(clusters[c])
            total_searched_vectors += len(candidate_indices)

            # Find nearest neighbor among candidates
            if len(candidate_indices) > 0:
                candidates = index_vectors[candidate_indices]
                distances = np.linalg.norm(candidates - q_vec, axis=1)
                nn_idx = np.argmin(distances)
                nn_label = index_labels[candidate_indices[nn_idx]]
                if nn_label == query_labels[q_idx]:
                    hits += 1

        total_queries = len(query_vectors)
        hit_rate = hits / total_queries if total_queries > 0 else 0
        hit_rates.append(hit_rate)

        penetration = total_searched_vectors / total_indexed
        x_labels.append(f"{clusters_to_search} ({penetration:.2f})")

        if highlight_point is None and hit_rate >= 0.98:
            highlight_point = (clusters_to_search, hit_rate, penetration)

        print(f"Clusters searched={clusters_to_search}, Hit rate={hit_rate:.4f}, Penetration={penetration:.2f}")

    return x_labels, hit_rates, highlight_point

# ---------------------
# Plotting function
# ---------------------
def plot_hit_rate_plotly(x_labels, hit_rates, highlight_point, title="KNN Clustering Hit Rate"):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x_labels,
        y=hit_rates,
        mode='lines+markers',
        name="Hit Rate",
        line=dict(width=2),
        marker=dict(size=6)
    ))

    if highlight_point:
        c_highlight, hr_highlight, penetration = highlight_point
        highlight_label = f"{c_highlight} ({penetration:.2f})"
        fig.add_trace(go.Scatter(
            x=[highlight_label],
            y=[hr_highlight],
            mode='markers+text',
            marker=dict(color='red', size=12, symbol='star'),
            text=[f"Penetration={penetration:.2f}"],
            textposition="top center",
            name="First ≥98% Hit"
        ))

    fig.update_layout(
        title=title,
        xaxis_title="Clusters searched (Penetration = vectors searched / total indexed)",
        yaxis_title="Hit Rate",
        template="plotly_white",
        hovermode="x unified"
    )
    fig.show()

# ---------------------
# Main workflow
# ---------------------
def main(json_path, n_clusters=14):
    x, y = load_data(json_path)
    index_x, index_y, query_x, query_y = split_representatives(x, y)

    kmeans, clusters = build_knn_clusters(index_x, n_clusters=n_clusters)

    x_labels, hit_rates, highlight_point = evaluate_hit_rate_knn(
        kmeans, clusters, index_x, index_y, query_x, query_y
    )

    plot_hit_rate_plotly(x_labels, hit_rates, highlight_point, title="KNN Clustering Hit Rate")

# ---------------------
# Run
# ---------------------
if __name__ == "__main__":
    main("iitd_features_1_with seprate_left_abd_wrute.json", n_clusters=14)


Clusters searched=1, Hit rate=0.7596, Penetration=140.63
Clusters searched=2, Hit rate=0.8681, Penetration=283.72
Clusters searched=3, Hit rate=0.9141, Penetration=421.35
Clusters searched=4, Hit rate=0.9263, Penetration=558.88
Clusters searched=5, Hit rate=0.9335, Penetration=693.27
Clusters searched=6, Hit rate=0.9385, Penetration=824.92
Clusters searched=7, Hit rate=0.9391, Penetration=956.24
Clusters searched=8, Hit rate=0.9407, Penetration=1085.07
Clusters searched=9, Hit rate=0.9407, Penetration=1213.70
Clusters searched=10, Hit rate=0.9418, Penetration=1343.53
Clusters searched=11, Hit rate=0.9429, Penetration=1469.64
Clusters searched=12, Hit rate=0.9429, Penetration=1581.14
Clusters searched=13, Hit rate=0.9429, Penetration=1689.71
Clusters searched=14, Hit rate=0.9429, Penetration=1805.00


In [6]:
!pip install scikit-learn


  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 42.8 MB/s eta 0:00:00a 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]


In [9]:
import json
import numpy as np
import plotly.graph_objects as go

# ---------------------
# Load JSON data
# ---------------------
def load_data(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    x = np.array(data["X"], dtype='float32')
    y = np.array(data["y"])
    return x, y

# ---------------------
# Split representatives & queries
# ---------------------
def split_representatives(x, y):
    unique_labels = np.unique(y)
    index_vectors = []
    index_labels = []
    query_vectors = []
    query_labels = []

    for label in unique_labels:
        indices = np.where(y == label)[0]
        if len(indices) == 0:
            continue
        index_vectors.append(x[indices[0]])
        index_labels.append(label)
        if len(indices) > 1:
            remaining = indices[1:]
            query_vectors.extend(x[remaining])
            query_labels.extend(y[remaining])

    return (np.array(index_vectors, dtype='float32'),
            np.array(index_labels),
            np.array(query_vectors, dtype='float32'),
            np.array(query_labels))

# ---------------------
# Random Hyperplane LSH
# ---------------------
class RandomHyperplaneLSH:
    def __init__(self, dim, nbits=16):
        self.nbits = nbits
        self.dim = dim
        self.planes = np.random.randn(nbits, dim).astype(np.float32)
        self.buckets = {}

    def _hash(self, vec):
        projections = vec.dot(self.planes.T)
        bits = (projections >= 0).astype(int)
        hash_code = int("".join(bits.astype(str)), 2)
        return hash_code

    def add(self, vectors):
        self.vectors = vectors
        self.n = vectors.shape[0]
        for idx, vec in enumerate(vectors):
            h = self._hash(vec)
            if h not in self.buckets:
                self.buckets[h] = []
            self.buckets[h].append(idx)

    def search_bucket(self, query_vec):
        h = self._hash(query_vec)
        return self.buckets.get(h, [])

# ---------------------
# Evaluate hit rate with top-k retrieval
# ---------------------
def evaluate_hit_rate_lsh_topk(index_vectors, index_labels, query_vectors, query_labels, nbits=16, top_k_max=50):
    dim = index_vectors.shape[1]
    lsh = RandomHyperplaneLSH(dim, nbits=nbits)
    lsh.add(index_vectors)
    total_indexed = len(index_labels)

    hit_rates = []
    x_labels = []
    highlight_point = None

    for k in range(1, top_k_max + 1):
        hits = 0
        total_retrieved = 0

        for q_idx, q_vec in enumerate(query_vectors):
            bucket_indices = lsh.search_bucket(q_vec)
            if len(bucket_indices) == 0:
                continue

            candidate_vecs = index_vectors[bucket_indices]
            distances = np.linalg.norm(candidate_vecs - q_vec, axis=1)
            sorted_idx = np.argsort(distances)[:k]
            selected_indices = [bucket_indices[i] for i in sorted_idx]

            total_retrieved += len(selected_indices)
            retrieved_labels = index_labels[selected_indices]
            if query_labels[q_idx] in retrieved_labels:
                hits += 1

        total_queries = len(query_vectors)
        hit_rate = hits / total_queries if total_queries > 0 else 0
        hit_rates.append(hit_rate)

        penetration = total_retrieved / total_indexed if total_indexed > 0 else 0
        x_labels.append(f"{k} ({penetration:.2f})")

        if highlight_point is None and hit_rate >= 0.98:
            highlight_point = (k, hit_rate, penetration)

        print(f"top_k={k}, Hit rate={hit_rate:.4f}, Penetration={penetration:.2f}")

    return x_labels, hit_rates, highlight_point

# ---------------------
# Plotting function
# ---------------------
def plot_hit_rate_plotly(x_labels, hit_rates, highlight_point, title="LSH Hit Rate"):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x_labels,
        y=hit_rates,
        mode='lines+markers',
        name="Hit Rate",
        line=dict(width=2),
        marker=dict(size=6)
    ))

    if highlight_point:
        k_highlight, hr_highlight, penetration = highlight_point
        highlight_label = f"{k_highlight} ({penetration:.2f})"
        fig.add_trace(go.Scatter(
            x=[highlight_label],
            y=[hr_highlight],
            mode='markers+text',
            marker=dict(color='red', size=12, symbol='star'),
            text=[f"Penetration={penetration:.2f}"],
            textposition="top center",
            name="First ≥98% Hit"
        ))

    fig.update_layout(
        title=title,
        xaxis_title="top_k (Penetration = vectors retrieved / total indexed)",
        yaxis_title="Hit Rate",
        template="plotly_white",
        hovermode="x unified"
    )
    fig.show()

# ---------------------
# Main workflow
# ---------------------
def main(json_path, nbits=16, top_k_max=50):
    x, y = load_data(json_path)
    index_x, index_y, query_x, query_y = split_representatives(x, y)

    x_labels, hit_rates, highlight_point = evaluate_hit_rate_lsh_topk(
        index_x, index_y, query_x, query_y, nbits=nbits, top_k_max=top_k_max
    )

    plot_hit_rate_plotly(x_labels, hit_rates, highlight_point, title=f"Random Hyperplane LSH Hit Rate ({nbits}-bits)")

# ---------------------
# Run
# ---------------------
if __name__ == "__main__":
    main("iitd_features_1_with seprate_left_abd_wrute.json", nbits=16, top_k_max=50)


top_k=1, Hit rate=0.2377, Penetration=3.49
top_k=2, Hit rate=0.2382, Penetration=6.51
top_k=3, Hit rate=0.2382, Penetration=9.16
top_k=4, Hit rate=0.2382, Penetration=11.51
top_k=5, Hit rate=0.2382, Penetration=13.76
top_k=6, Hit rate=0.2382, Penetration=15.86
top_k=7, Hit rate=0.2388, Penetration=17.73
top_k=8, Hit rate=0.2388, Penetration=19.46
top_k=9, Hit rate=0.2393, Penetration=21.11
top_k=10, Hit rate=0.2393, Penetration=22.72
top_k=11, Hit rate=0.2393, Penetration=24.33
top_k=12, Hit rate=0.2393, Penetration=25.84
top_k=13, Hit rate=0.2393, Penetration=27.21
top_k=14, Hit rate=0.2393, Penetration=28.59
top_k=15, Hit rate=0.2393, Penetration=29.76
top_k=16, Hit rate=0.2393, Penetration=30.80
top_k=17, Hit rate=0.2393, Penetration=31.84
top_k=18, Hit rate=0.2393, Penetration=32.79
top_k=19, Hit rate=0.2393, Penetration=33.74
top_k=20, Hit rate=0.2393, Penetration=34.52
top_k=21, Hit rate=0.2393, Penetration=35.31
top_k=22, Hit rate=0.2393, Penetration=36.09
top_k=23, Hit rate=0.2

In [10]:
import json
import numpy as np
from sklearn.cluster import KMeans
import plotly.graph_objects as go
import os

# --- Constants ---
DEFAULT_N_CLUSTERS = 10
HIGHLIGHT_THRESHOLD = 0.98

def generate_synthetic_data(n_classes=10, samples_per_class=20, dim=64):
    """
    Generates synthetic clustered data to simulate feature vectors.

    Args:
        n_classes (int): The number of unique classes (e.g., individuals).
        samples_per_class (int): The number of samples for each class.
        dim (int): The dimensionality of the feature vectors.

    Returns:
        tuple: A tuple containing the feature vectors (X) and labels (y).
    """
    np.random.seed(42)
    centroids = np.random.rand(n_classes, dim) * 10
    X = []
    y = []
    for i in range(n_classes):
        # Add noise to centroids to create clusters
        points = centroids[i] + np.random.randn(samples_per_class, dim) * 0.5
        X.extend(points)
        y.extend([i] * samples_per_class)
    
    return np.array(X, dtype='float32'), np.array(y)


def save_data(X, y, json_path):
    """Saves the feature vectors and labels to a JSON file."""
    with open(json_path, 'w') as f:
        json.dump({"X": X.tolist(), "y": y.tolist()}, f, indent=4)


def load_data(json_path):
    """
    Loads feature vectors and labels from a JSON file.
    This simulates loading the output from a feature extraction model like IrisIndexNet.
    """
    with open(json_path, 'r') as f:
        data = json.load(f)
    x = np.array(data["X"], dtype='float32')
    y = np.array(data["y"])
    return x, y


def split_representatives(x, y):
    """
    Splits the dataset into index vectors (for building the search index)
    and query vectors (for testing the index).

    The first vector of each class is used for the index, and the rest are used as queries.
    """
    unique_labels = np.unique(y)
    index_vectors = []
    index_labels = []
    query_vectors = []
    query_labels = []

    for label in unique_labels:
        indices = np.where(y == label)[0]
        if len(indices) == 0:
            continue
        
        # First sample of each class becomes a representative in the index
        index_vectors.append(x[indices[0]])
        index_labels.append(label)

        # Remaining samples of that class become queries
        if len(indices) > 1:
            remaining_indices = indices[1:]
            query_vectors.extend(x[remaining_indices])
            query_labels.extend(y[remaining_indices])

    return (
        np.array(index_vectors, dtype='float32'),
        np.array(index_labels),
        np.array(query_vectors, dtype='float32'),
        np.array(query_labels)
    )


def build_kmeans_clusters(vectors, n_clusters=DEFAULT_N_CLUSTERS):
    """
    Builds K-Means clusters from the index vectors. This creates the 'Index Table'.
    Each cluster centroid acts as an index.
    """
    print(f"\nBuilding {n_clusters} K-Means clusters...")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans.fit(vectors)
    
    # Create a mapping from cluster ID to the indices of vectors in that cluster
    cluster_assignments = kmeans.labels_
    clusters = {i: [] for i in range(n_clusters)}
    for vector_idx, cluster_id in enumerate(cluster_assignments):
        clusters[cluster_id].append(vector_idx)
        
    print("Clustering complete.")
    return kmeans, clusters


def evaluate_hit_rate(kmeans, clusters, index_vectors, index_labels, query_vectors, query_labels, max_clusters=None):
    """
    Evaluates the performance of the index by simulating queries.
    It calculates the hit rate and penetration rate for an increasing number of searched clusters.
    """
    hit_rates = []
    x_axis_labels = []
    highlight_point = None
    total_indexed_vectors = len(index_labels)
    
    n_clusters = len(clusters)
    if max_clusters is None:
        max_clusters = n_clusters

    print("\nEvaluating Hit Rate vs. Penetration Rate...")
    for clusters_to_search in range(1, max_clusters + 1):
        hits = 0
        total_searched_vectors_for_this_run = 0

        for q_idx, q_vec in enumerate(query_vectors):
            # 1. Find the nearest cluster centroids for the query vector
            centroid_dists = np.linalg.norm(kmeans.cluster_centers_ - q_vec, axis=1)
            nearest_cluster_ids = np.argsort(centroid_dists)[:clusters_to_search]

            # 2. Collect all vectors from these clusters to form the candidate list
            candidate_indices = []
            for c_id in nearest_cluster_ids:
                candidate_indices.extend(clusters.get(c_id, []))
            
            total_searched_vectors_for_this_run += len(candidate_indices)

            # 3. Search within the candidate list to find the nearest neighbor
            if len(candidate_indices) > 0:
                candidate_vecs = index_vectors[candidate_indices]
                
                # Find the single closest vector within the candidate set
                distances = np.linalg.norm(candidate_vecs - q_vec.reshape(1, -1), axis=1)
                nn_in_candidate_idx = np.argmin(distances)
                
                # Get the original index and label of this nearest neighbor
                original_nn_idx = candidate_indices[nn_in_candidate_idx]
                nn_label = index_labels[original_nn_idx]
                
                # Check if it's a "hit"
                if nn_label == query_labels[q_idx]:
                    hits += 1

        # Calculate metrics for this number of searched clusters
        total_queries = len(query_vectors)
        hit_rate = hits / total_queries if total_queries > 0 else 0
        
        # Penetration rate: the average fraction of the database searched per query
        penetration_rate = (total_searched_vectors_for_this_run / total_queries) / total_indexed_vectors if total_queries > 0 and total_indexed_vectors > 0 else 0
        
        hit_rates.append(hit_rate)
        x_axis_labels.append(f"{clusters_to_search}<br>({penetration_rate:.1%})")

        # Store the first point that crosses the highlight threshold
        if highlight_point is None and hit_rate >= HIGHLIGHT_THRESHOLD:
            highlight_point = (clusters_to_search, hit_rate, penetration_rate)

        print(f"Clusters Searched: {clusters_to_search:2d}, Hit Rate: {hit_rate:.4f}, Avg Penetration: {penetration_rate:.2%}")

    return x_axis_labels, hit_rates, highlight_point


def plot_results(x_labels, hit_rates, highlight_point, title="K-Means Indexing Performance"):
    """
    Generates an interactive plot of Hit Rate vs. Penetration Rate using Plotly.
    This visualizes the trade-off between accuracy and search speed.
    """
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x_labels,
        y=hit_rates,
        mode='lines+markers',
        name="Hit Rate",
        line=dict(width=2, color='#1f77b4'),
        marker=dict(size=8)
    ))

    # Add a special marker for the highlight point
    if highlight_point:
        clusters, hr, pen = highlight_point
        highlight_label = f"{clusters}<br>({pen:.1%})"
        fig.add_trace(go.Scatter(
            x=[highlight_label],
            y=[hr],
            mode='markers+text',
            marker=dict(color='red', size=14, symbol='star'),
            text=[f"Hit Rate: {hr:.2%}<br>Penetration: {pen:.2%}"],
            textposition="top right",
            name=f"First to reach {HIGHLIGHT_THRESHOLD*100}% Hit Rate"
        ))

    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=20)),
        xaxis_title="Clusters Searched (Average Penetration Rate)",
        yaxis_title="Hit Rate",
        yaxis=dict(range=[min(hit_rates) * 0.98 if hit_rates else 0, 1.01], tickformat=".0%"),
        template="plotly_white",
        hovermode="x unified",
        legend=dict(x=0.01, y=0.99, bordercolor="Black", borderwidth=1),
        font=dict(family="Arial, sans-serif", size=12)
    )
    fig.show()


def main(json_path, n_clusters=DEFAULT_N_CLUSTERS):
    """Main workflow to run the simulation."""
    # Step 0: Ensure data exists. If not, create synthetic data.
    if not os.path.exists(json_path):
        print(f"'{json_path}' not found. Generating synthetic sample data.")
        X_gen, y_gen = generate_synthetic_data()
        save_data(X_gen, y_gen, json_path)
        print(f"Sample data saved to '{json_path}'.")

    # Step 1: Load pre-computed feature vectors
    x, y = load_data(json_path)

    # Step 2: Split data into a set for indexing and a set for querying
    index_x, index_y, query_x, query_y = split_representatives(x, y)
    print(f"Data loaded: {len(index_x)} vectors for indexing, {len(query_x)} for querying.")

    # Step 3: Build the K-Means clusters (the index)
    kmeans, clusters = build_kmeans_clusters(index_x, n_clusters=n_clusters)

    # Step 4: Evaluate the index's performance
    x_labels, hit_rates, highlight = evaluate_hit_rate(
        kmeans, clusters, index_x, index_y, query_x, query_y
    )

    # Step 5: Plot the results
    plot_results(x_labels, hit_rates, highlight)


if __name__ == "__main__":
    # --- Configuration ---
    # The JSON file containing feature vectors and labels.
    # A sample file will be generated if this does not exist.
    DATA_FILE = "iitd_features_1_with seprate_left_abd_wrute.json"
    
    # The number of clusters to create in the index.
    # This corresponds to the number of entries in the "Index Table".
    NUM_CLUSTERS = 14
    
    main(DATA_FILE, n_clusters=NUM_CLUSTERS)

Data loaded: 435 vectors for indexing, 1805 for querying.

Building 14 K-Means clusters...
Clustering complete.

Evaluating Hit Rate vs. Penetration Rate...
Clusters Searched:  1, Hit Rate: 0.7596, Avg Penetration: 7.79%
Clusters Searched:  2, Hit Rate: 0.8681, Avg Penetration: 15.72%
Clusters Searched:  3, Hit Rate: 0.9141, Avg Penetration: 23.34%
Clusters Searched:  4, Hit Rate: 0.9263, Avg Penetration: 30.96%
Clusters Searched:  5, Hit Rate: 0.9335, Avg Penetration: 38.41%
Clusters Searched:  6, Hit Rate: 0.9385, Avg Penetration: 45.70%
Clusters Searched:  7, Hit Rate: 0.9391, Avg Penetration: 52.98%
Clusters Searched:  8, Hit Rate: 0.9407, Avg Penetration: 60.11%
Clusters Searched:  9, Hit Rate: 0.9407, Avg Penetration: 67.24%
Clusters Searched: 10, Hit Rate: 0.9418, Avg Penetration: 74.43%
Clusters Searched: 11, Hit Rate: 0.9429, Avg Penetration: 81.42%
Clusters Searched: 12, Hit Rate: 0.9429, Avg Penetration: 87.60%
Clusters Searched: 13, Hit Rate: 0.9429, Avg Penetration: 93.61%

In [11]:
import json
import os
import numpy as np
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go
import warnings

# Suppress ConvergenceWarning for KMeans to keep the output clean
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

def load_features_from_user_json(json_path):
    """
    Loads feature vectors and labels directly from the user-provided JSON file.
    It expects the keys to be "X" for features and "y" for labels.
    """
    if not os.path.exists(json_path):
        print(f"Error: The specified JSON file was not found at '{json_path}'")
        print("Please make sure the file is in the same directory as this script.")
        exit()
        
    print(f"Loading features and labels from '{json_path}'...")
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # Use the keys "X" and "y" as specified in your initial code
    features = np.array(data["X"], dtype='float32')
    labels = np.array(data["y"])
    
    print(f"Successfully loaded {len(features)} vectors with {len(np.unique(labels))} unique classes.")
    return features, labels

def split_gallery_probe(features, labels, probe_ratio=0.2):
    """
    Splits the data into a gallery (for indexing) and a probe (for querying) set.
    This follows the 80-20 split methodology from the paper.
    """
    print(f"\nSplitting data into gallery ({(1-probe_ratio)*100:.0f}%) and probe ({probe_ratio*100:.0f}%) sets.")
    indices = np.arange(len(labels))
    gallery_idx, probe_idx = train_test_split(
        indices, test_size=probe_ratio, random_state=42, stratify=labels
    )
    
    gallery_features = features[gallery_idx]
    gallery_labels = labels[gallery_idx]
    probe_features = features[probe_idx]
    probe_labels = labels[probe_idx]
    
    print(f"Gallery size: {len(gallery_features)}, Probe size: {len(probe_features)}")
    return gallery_features, gallery_labels, probe_features, probe_labels

def build_kmeans_index(gallery_features, n_clusters):
    """
    Builds the index table using K-Means++ clustering.
    """
    print(f"\nBuilding K-Means index with {n_clusters} clusters...")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_assignments = kmeans.fit_predict(gallery_features)
    
    index_table = {i: [] for i in range(n_clusters)}
    for feature_idx, cluster_id in enumerate(cluster_assignments):
        index_table[cluster_id].append(feature_idx)
        
    print("K-Means indexing complete.")
    return index_table, kmeans.cluster_centers_

def build_agglomerative_index(gallery_features, n_clusters):
    """
    Builds the index table using Agglomerative Clustering.
    """
    print(f"\nBuilding Agglomerative index with {n_clusters} clusters...")
    agg_cluster = AgglomerativeClustering(n_clusters=n_clusters, metric='euclidean', linkage='ward')
    cluster_assignments = agg_cluster.fit_predict(gallery_features)

    index_table = {i: [] for i in range(n_clusters)}
    for feature_idx, cluster_id in enumerate(cluster_assignments):
        index_table[cluster_id].append(feature_idx)
        
    print("Agglomerative indexing complete.")
    return index_table

def evaluate_retrieval(probe_features, probe_labels, gallery_features, gallery_labels, index_table, cluster_centers=None):
    """
    Evaluates the performance of an index table by calculating Hit Rate vs. Penetration Rate.
    """
    results = {'penetration_rates': [], 'hit_rates': []}
    n_total_clusters = len(index_table)
    total_gallery_size = len(gallery_features)

    print(f"\nEvaluating retrieval performance (searching from 1 to {n_total_clusters} clusters)...")
    for n_clusters_to_search in range(1, n_total_clusters + 1):
        total_hits = 0
        total_searched_count = 0

        for i in range(len(probe_features)):
            probe_vec = probe_features[i]
            true_label = probe_labels[i]

            if cluster_centers is not None: # K-Means uses pre-computed centroids
                dists = np.linalg.norm(cluster_centers - probe_vec, axis=1)
            else: # Agglomerative must compute mean distance to clusters on the fly
                dists = np.array([
                    np.linalg.norm(gallery_features[index_table[c_id]] - probe_vec, axis=1).mean()
                    for c_id in range(n_total_clusters) if c_id in index_table and len(index_table[c_id]) > 0
                ])
            
            nearest_cluster_ids = np.argsort(dists)[:n_clusters_to_search]

            candidate_indices = []
            for c_id in nearest_cluster_ids:
                candidate_indices.extend(index_table.get(c_id, []))
            
            total_searched_count += len(candidate_indices)

            if len(candidate_indices) > 0:
                candidate_gallery_labels = gallery_labels[candidate_indices]
                search_dists = np.linalg.norm(gallery_features[candidate_indices] - probe_vec, axis=1)
                predicted_label = candidate_gallery_labels[np.argmin(search_dists)]

                if predicted_label == true_label:
                    total_hits += 1
        
        hit_rate = total_hits / len(probe_features)
        penetration_rate = (total_searched_count / len(probe_features)) / total_gallery_size
        
        results['hit_rates'].append(hit_rate)
        results['penetration_rates'].append(penetration_rate)

    print("Evaluation complete.")
    return results

def plot_performance_curves(results_dict, filename):
    """
    Plots the Hit Rate vs. Penetration Rate curves for all evaluated methods.
    """
    fig = go.Figure()
    colors = {'K-Means++': '#d62728', 'Agglomerative': '#1f77b4'}
    
    for method, results in results_dict.items():
        fig.add_trace(go.Scatter(
            x=results['penetration_rates'],
            y=results['hit_rates'],
            mode='lines+markers',
            name=method,
            line=dict(width=2, color=colors.get(method)),
            marker=dict(size=6)
        ))

    fig.update_layout(
        title=f"Indexing Performance Comparison on '{filename}'",
        xaxis_title="Penetration Rate (%)",
        yaxis_title="Hit Rate (%)",
        xaxis=dict(tickformat=".1%"),
        yaxis=dict(tickformat=".0%"),
        template="plotly_white",
        legend=dict(x=0.98, y=0.02, xanchor='right', yanchor='bottom'),
        font=dict(family="Arial")
    )
    fig.show()

def main():
    """
    Executes the full pipeline using the user-provided JSON file.
    """
    # --- Configuration ---
    # !! IMPORTANT !!
    # Place your JSON file in the same directory as this script.
    USER_JSON_FILE = "iitd_features_1_with seprate_left_abd_wrute.json"
    
    # You can tune these parameters based on your dataset's characteristics.
    # These are reasonable starting points.
    N_CLUSTERS_KMEANS = 15
    N_CLUSTERS_AGG = 20
    
    # 1. Load your features from your JSON file
    features, labels = load_features_from_user_json(USER_JSON_FILE)
    
    # 2. Split into Gallery (for index) and Probe (for query)
    gallery_features, gallery_labels, probe_features, probe_labels = split_gallery_probe(features, labels)

    # --- Run K-Means Pipeline ---
    kmeans_index, kmeans_centers = build_kmeans_index(gallery_features, n_clusters=N_CLUSTERS_KMEANS)
    kmeans_results = evaluate_retrieval(
        probe_features, probe_labels, gallery_features, gallery_labels, kmeans_index, kmeans_centers
    )

    # --- Run Agglomerative Pipeline ---
    agg_index = build_agglomerative_index(gallery_features, n_clusters=N_CLUSTERS_AGG)
    agg_results = evaluate_retrieval(
        probe_features, probe_labels, gallery_features, gallery_labels, agg_index
    )

    # 4. Plot comparative results
    plot_performance_curves({
        'K-Means++': kmeans_results,
        'Agglomerative': agg_results
    }, USER_JSON_FILE)

if __name__ == "__main__":
    main()


Loading features and labels from 'iitd_features_1_with seprate_left_abd_wrute.json'...
Successfully loaded 2240 vectors with 435 unique classes.

Splitting data into gallery (80%) and probe (20%) sets.
Gallery size: 1792, Probe size: 448

Building K-Means index with 15 clusters...
K-Means indexing complete.

Evaluating retrieval performance (searching from 1 to 15 clusters)...
Evaluation complete.

Building Agglomerative index with 20 clusters...
Agglomerative indexing complete.

Evaluating retrieval performance (searching from 1 to 20 clusters)...
Evaluation complete.


In [13]:
import json
import os
import numpy as np
import faiss  # Import the FAISS library
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

# --- 1. DATA LOADING & SPLITTING ---
def load_features_from_user_json(json_path):
    if not os.path.exists(json_path):
        print(f"Error: JSON file not found at '{json_path}'")
        exit()
    print(f"Loading features from '{json_path}'...")
    with open(json_path, 'r') as f:
        data = json.load(f)
    features = np.array(data["X"], dtype='float32')
    labels = np.array(data["y"])
    print(f"Loaded {len(features)} vectors.")
    return features, labels

def split_gallery_probe(features, labels, probe_ratio=0.2):
    print(f"\nSplitting data into gallery ({(1-probe_ratio)*100:.0f}%) and probe ({probe_ratio*100:.0f}%) sets.")
    indices = np.arange(len(labels))
    gallery_idx, probe_idx = train_test_split(
        indices, test_size=probe_ratio, random_state=42, stratify=labels)
    gallery_features = features[gallery_idx]
    gallery_labels = labels[gallery_idx]
    probe_features = features[probe_idx]
    probe_labels = labels[probe_idx]
    print(f"Gallery size: {len(gallery_features)}, Probe size: {len(probe_features)}")
    return gallery_features, gallery_labels, probe_features, probe_labels

# --- 2. INDEXING TECHNIQUES ---

# --- Method A: Standard K-Means ---
def build_kmeans_index(gallery_features, n_clusters):
    print(f"\nBuilding Standard K-Means index with {n_clusters} clusters...")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_assignments = kmeans.fit_predict(gallery_features)
    index_table = {i: [] for i in range(n_clusters)}
    for feature_idx, cluster_id in enumerate(cluster_assignments):
        index_table[cluster_id].append(feature_idx)
    print("K-Means indexing complete.")
    return index_table, kmeans

# --- Method B: Hybrid Cluster Hashing (HCH) ---
class HybridClusterHashIndex:
    def __init__(self, n_clusters, n_lsh_bits):
        self.n_clusters = n_clusters
        self.n_lsh_bits = n_lsh_bits
        self.kmeans = None
        self.lsh_indices = {}
        self.cluster_to_original_indices = {}

    def build(self, gallery_features):
        print(f"\nBuilding Hybrid Cluster Hash (HCH) Index...")
        print(f"Step 1: Performing coarse clustering with {self.n_clusters} K-Means clusters.")
        feature_dim = gallery_features.shape[1]
        
        self.kmeans = KMeans(n_clusters=self.n_clusters, random_state=42, n_init=10)
        cluster_assignments = self.kmeans.fit_predict(gallery_features)

        print(f"Step 2: Building {self.n_clusters} fine-grained FAISS LSH indices...")
        for i in range(self.n_clusters):
            cluster_mask = (cluster_assignments == i)
            self.cluster_to_original_indices[i] = np.where(cluster_mask)[0]
            cluster_vectors = gallery_features[cluster_mask]
            
            if len(cluster_vectors) == 0:
                continue

            index = faiss.IndexLSH(feature_dim, self.n_lsh_bits)
            index.add(cluster_vectors)
            self.lsh_indices[i] = index
        print("HCH Index build complete.")

    def search_in_clusters(self, query_vector, cluster_ids, k_candidates):
        query_vector_2d = np.array([query_vector])
        all_original_indices = []
        
        for cluster_id in cluster_ids:
            if cluster_id not in self.lsh_indices:
                continue

            lsh_index = self.lsh_indices[cluster_id]
            num_items_in_cluster = lsh_index.ntotal
            actual_k = min(k_candidates, num_items_in_cluster)

            if actual_k == 0:
                continue
            
            _, local_indices = lsh_index.search(query_vector_2d, actual_k)
            original_indices = self.cluster_to_original_indices[cluster_id][local_indices[0]]
            all_original_indices.extend(original_indices)
            
        return list(set(all_original_indices))

# --- 3. EVALUATION ---
def evaluate_standard_clustering(probe_features, probe_labels, gallery_features, gallery_labels, index_table, kmeans_model):
    """Evaluates performance for standard clustering by searching an increasing number of clusters."""
    results = {'penetration_rates': [], 'hit_rates': []}
    n_total_clusters = kmeans_model.n_clusters
    total_gallery_size = len(gallery_features)
    print(f"\nEvaluating standard clustering (searching 1 to {n_total_clusters} clusters)...")
    
    for n_clusters_to_search in range(1, n_total_clusters + 1):
        total_hits, total_searched_count = 0, 0
        for i in range(len(probe_features)):
            probe_vec, true_label = probe_features[i], probe_labels[i]
            
            dists = np.linalg.norm(kmeans_model.cluster_centers_ - probe_vec, axis=1)
            nearest_cluster_ids = np.argsort(dists)[:n_clusters_to_search]
            
            candidate_indices = [idx for c_id in nearest_cluster_ids for idx in index_table.get(c_id, [])]
            total_searched_count += len(candidate_indices)

            if len(candidate_indices) > 0:
                search_dists = np.linalg.norm(gallery_features[candidate_indices] - probe_vec, axis=1)
                predicted_label = gallery_labels[candidate_indices[np.argmin(search_dists)]]
                if predicted_label == true_label:
                    total_hits += 1
        
        hit_rate = total_hits / len(probe_features)
        penetration_rate = (total_searched_count / len(probe_features)) / total_gallery_size
        results['hit_rates'].append(hit_rate)
        results['penetration_rates'].append(penetration_rate)
    print("Evaluation complete.")
    return results

def evaluate_hch_retrieval(hch_index, probe_features, probe_labels, gallery_features, gallery_labels):
    """Evaluates the performance of our HCH index by searching an increasing number of coarse clusters."""
    results = {'penetration_rates': [], 'hit_rates': []}
    total_gallery_size = len(gallery_features)
    CANDIDATES_PER_CLUSTER = 10 
    
    print(f"\nEvaluating HCH retrieval (searching 1 to {hch_index.n_clusters} clusters)...")

    for n_clusters_to_search in range(1, hch_index.n_clusters + 1):
        total_hits, total_searched_count = 0, 0
        
        for i in range(len(probe_features)):
            probe_vec, true_label = probe_features[i], probe_labels[i]
            
            dists_to_centroids = np.linalg.norm(hch_index.kmeans.cluster_centers_ - probe_vec, axis=1)
            closest_cluster_ids = np.argsort(dists_to_centroids)[:n_clusters_to_search]
            
            candidate_indices = hch_index.search_in_clusters(probe_vec, closest_cluster_ids, k_candidates=CANDIDATES_PER_CLUSTER)
            total_searched_count += len(candidate_indices)

            if len(candidate_indices) > 0:
                candidate_labels = gallery_labels[candidate_indices]
                if true_label in candidate_labels:
                    total_hits += 1

        hit_rate = total_hits / len(probe_features)
        penetration_rate = (total_searched_count / len(probe_features)) / total_gallery_size
        results['hit_rates'].append(hit_rate)
        results['penetration_rates'].append(penetration_rate)
        print(f"  [HCH] Clusters searched: {n_clusters_to_search}, Hit Rate: {hit_rate:.2%}, Penetration Rate: {penetration_rate:.2%}")

    print("Evaluation complete.")
    return results

# --- 4. VISUALIZATION ---
def plot_performance_curves(results_dict, filename):
    fig = go.Figure()
    colors = {
        'Hybrid Cluster Hash (HCH)': '#e377c2',
        'Standard K-Means': '#1f77b4',
    }
    
    for method, results in results_dict.items():
        fig.add_trace(go.Scatter(
            x=results['penetration_rates'], y=results['hit_rates'],
            mode='lines+markers', name=method,
            line=dict(width=3, color=colors.get(method)), marker=dict(size=8)
        ))

    fig.update_layout(
        title=f"Performance of Invented HCH Index vs. Standard Methods on '{filename}'",
        xaxis_title="Penetration Rate (Avg. % of Gallery Searched)",
        yaxis_title="Hit Rate",
        xaxis=dict(tickformat=".2%"), yaxis=dict(tickformat=".0%"),
        template="plotly_white",
        legend=dict(x=0.98, y=0.02, xanchor='right', yanchor='bottom'),
        font=dict(family="Arial", size=12)
    )
    fig.show()

# --- 5. MAIN WORKFLOW ---
def main():
    USER_JSON_FILE = "iitd_features_1_with seprate_left_abd_wrute.json"
    
    # --- Config for Standard K-Means ---
    N_CLUSTERS_KMEANS = 15

    # --- Config for our HCH method ---
    N_COARSE_CLUSTERS_HCH = 5 
    N_LSH_BITS = 16 

    features, labels = load_features_from_user_json(USER_JSON_FILE)
    gallery_features, gallery_labels, probe_features, probe_labels = split_gallery_probe(features, labels)

    # --- Pipeline 1: Standard K-Means ---
    kmeans_index, kmeans_model = build_kmeans_index(gallery_features, n_clusters=N_CLUSTERS_KMEANS)
    kmeans_results = evaluate_standard_clustering(probe_features, probe_labels, gallery_features, gallery_labels, kmeans_index, kmeans_model)

    # --- Pipeline 2: Our new HCH Index ---
    hch_index = HybridClusterHashIndex(n_clusters=N_COARSE_CLUSTERS_HCH, n_lsh_bits=N_LSH_BITS)
    hch_index.build(gallery_features)
    hch_results = evaluate_hch_retrieval(hch_index, probe_features, probe_labels, gallery_features, gallery_labels)
    
    # --- Plot the results for comparison ---
    plot_performance_curves({
        'Standard K-Means': kmeans_results,
        'Hybrid Cluster Hash (HCH)': hch_results,
    }, USER_JSON_FILE)

if __name__ == "__main__":
    main()



Loading features from 'iitd_features_1_with seprate_left_abd_wrute.json'...
Loaded 2240 vectors.

Splitting data into gallery (80%) and probe (20%) sets.
Gallery size: 1792, Probe size: 448

Building Standard K-Means index with 15 clusters...
K-Means indexing complete.

Evaluating standard clustering (searching 1 to 15 clusters)...
Evaluation complete.

Building Hybrid Cluster Hash (HCH) Index...
Step 1: Performing coarse clustering with 5 K-Means clusters.
Step 2: Building 5 fine-grained FAISS LSH indices...
HCH Index build complete.

Evaluating HCH retrieval (searching 1 to 5 clusters)...
  [HCH] Clusters searched: 1, Hit Rate: 65.62%, Penetration Rate: 0.56%
  [HCH] Clusters searched: 2, Hit Rate: 74.78%, Penetration Rate: 1.12%
  [HCH] Clusters searched: 3, Hit Rate: 76.56%, Penetration Rate: 1.67%
  [HCH] Clusters searched: 4, Hit Rate: 77.01%, Penetration Rate: 2.23%
  [HCH] Clusters searched: 5, Hit Rate: 77.23%, Penetration Rate: 2.79%
Evaluation complete.
